# Memory Concepts (메모리 개념)

**메모리(Memory)** 는 AI 에이전트가 대화 이력과 상태를 저장하고 관리하는 시스템입니다.

메모리는 두 가지 주요 유형으로 분류됩니다:
- **단기 메모리(Short-term Memory)**: 하나의 대화 세션(thread) 내에서 이전 상호작용을 기억
- **장기 메모리(Long-term Memory)**: 여러 대화 세션에 걸쳐 정보를 유지

이 노트북에서는 메모리의 기본 개념과 사용법을 다룹니다.

**참고**: [LangChain 공식 문서 - Memory](https://docs.langchain.com/oss/python/concepts/memory)

In [ ]:
# LangSmith 추적 (선택적 - API 키가 있을 때만 활성화)

## 1. 단기 메모리 (Short-term Memory)

단기 메모리는 하나의 대화 세션(thread) 내에서 이전 메시지를 기억합니다.
에이전트가 대화의 문맥을 유지하고 이전 대화 내용을 참조할 수 있게 합니다.

### 1.1 Checkpointer 없이 사용

Checkpointer 없이는 각 호출이 독립적이며, 이전 대화를 기억하지 못합니다.

In [ ]:
def search_db(query: str) -> str:
# Checkpointer 없는 에이전트
# 첫 번째 대화
# 두 번째 대화 - 이전 내용을 기억하지 못함

### 1.2 InMemorySaver로 단기 메모리 추가

`InMemorySaver`는 메모리에 대화 이력을 저장합니다.
프로세스가 종료되면 데이터가 사라집니다.

In [ ]:
# InMemorySaver를 사용한 에이전트
# 대화 설정 (thread_id로 대화 식별)
# 첫 번째 대화
# 두 번째 대화 - 같은 thread_id로 이전 내용을 기억

### 1.3 대화 세션 분리

다른 `thread_id`를 사용하면 별도의 대화 세션이 됩니다.

In [ ]:
# 새로운 대화 세션

## 2. 영구 저장 (Persistent Storage)

`SqliteSaver`를 사용하면 대화 이력을 데이터베이스에 영구 저장할 수 있습니다.
프로세스가 재시작되어도 데이터가 유지됩니다.

### 2.1 SqliteSaver 사용

In [ ]:
# SQLite 데이터베이스 파일 경로
# SqliteSaver 생성 및 사용
    # 테이블 자동 생성
    # 에이전트 생성
    # 대화 진행
# 데이터베이스에서 다시 로드
    # 같은 thread_id로 이전 대화 이어가기

## 3. 대화 이력 조회

Checkpointer를 통해 저장된 대화 이력을 조회할 수 있습니다.

In [ ]:
# 여러 대화 진행
# 최종 응답 확인
# 전체 메시지 이력 확인

## 4. 장기 메모리 개념 (Long-term Memory)

장기 메모리는 여러 대화 세션에 걸쳐 유지되는 정보입니다.
사용자 선호도, 이전 대화 요약, 학습한 정보 등을 저장합니다.

장기 메모리 구현 방법:
- **Checkpointer 기반**: PostgreSQL, SQLite 등을 통한 대화 이력 저장
- **Store 기반**: JSON 문서 형식으로 사용자 정보, 설정 등 저장
- **벡터 데이터베이스**: 임베딩을 활용한 의미 기반 검색

### 4.1 Checkpointer를 통한 여러 세션 정보 유지

In [ ]:
# 첫 번째 세션: 정보 저장
# 두 번째 세션: 이전 정보 활용
    # 같은 사용자의 새로운 대화

### 4.2 Store 기반 장기 메모리

`Store`는 JSON 문서 형식으로 사용자 정보, 설정 등을 저장하는 장기 메모리입니다.
Checkpointer와 달리 대화 이력이 아닌 구조화된 데이터를 저장합니다.

주요 개념:
- **namespace**: 데이터를 그룹화하는 단위 (예: 사용자별, 조직별)
- **key**: 각 메모리를 구분하는 고유 식별자
- **value**: 저장할 JSON 문서

In [ ]:
# Store 생성
# 사용자 정보 저장 예시
# 저장된 정보 조회

### 4.3 도구에서 Store 사용하기

도구에서 Store를 통해 장기 메모리에 접근할 수 있습니다.

In [ ]:
class Context:
def get_user_preferences(runtime: ToolRuntime[Context, AgentState]) -> str:
# Store를 사용하는 에이전트 생성
# 사용자 선호도 조회

## 5. 메모리 전략

효과적인 메모리 관리를 위한 전략:

### 개발 단계
- **InMemorySaver**: 빠른 프로토타이핑과 테스트
- 데이터 손실이 문제되지 않는 경우

### 프로덕션 단계
- **SqliteSaver**: 소규모 애플리케이션, 단일 인스턴스
- **PostgresSaver**: 대규모 애플리케이션, 다중 인스턴스
- 백업 및 복구 전략 필요

### 메모리 최적화
- 오래된 대화 아카이빙
- 중요 정보만 요약하여 저장
- 컨텍스트 윈도우 제한 고려

## 6. 실전 예제: 사용자 컨텍스트 유지

In [ ]:
def get_recommendation(category: str) -> str:
# 선호도 학습
# 컨텍스트 기반 추천

## 주요 포인트 정리

1. **InMemorySaver**: 개발/테스트용, 프로세스 종료 시 데이터 손실
2. **SqliteSaver**: 영구 저장, 소규모 애플리케이션에 적합
3. **Thread ID**: 대화 세션을 구분하는 고유 식별자
4. **Checkpointer**: 대화 상태를 저장하고 복원하는 메커니즘 (대화 이력 저장)
5. **Store**: JSON 문서 형식으로 사용자 정보, 설정 등을 저장 (구조화된 데이터 저장)
6. **장기 메모리**: 여러 세션에 걸친 정보 유지

**다음 단계**: 
- [110_Semantic_Search.py](110_Semantic_Search.py)에서 검색 엔진 구축 학습
- 메모리와 검색을 결합한 RAG 에이전트 구현